In [1]:
import os
import torch
import random
import torch.nn as nn
import torch.backends.cudnn as cudnn
from models import build_model
import numpy as np
from PIL import Image

/opt/anaconda3/envs/LaVIT-main/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")


Please 'pip install apex'
Please 'pip install apex'
Please 'pip install apex'
Please 'pip install apex'


In [2]:
# The local directory to save LaVIT checkpoint, set to yours
model_path = "/data/phd/wsun/checkpoint_LaVIT/VideoLaVIT-v1/language_model_sft"   # 例：/home/jinyang06/models/VideoLaVIT-v1/language_model_sft
model_dtype='bf16'

max_video_clips = 16
device_id = 0
torch.cuda.set_device(device_id)
device = torch.device('cuda')

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# For Multi-Modal Understanding
runner = build_model(model_path=model_path, model_dtype=model_dtype, understanding=True,
        device_id=device_id, use_xformers=True, max_video_clips=max_video_clips,)
print("Building Model Finsished")

Loading Video LaVIT Model Weight from /data/phd/wsun/checkpoint_LaVIT/VideoLaVIT-v1/language_model_sft, model precision: bf16
Not used {}


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at /data/phd/wsun/checkpoint_LaVIT/VideoLaVIT-v1/language_model_sft were not used when initializing VideoLaVITLlamaForCausalLM: ['model.motion_tokenizer.quantize.embedding.cluster_size', 'model.motion_tokenizer.quantize.cluster_size', 'model.motion_tokenizer.quantize.embedding.embed_avg', 'model.motion_tokenizer.quantize.embedding.initted']
- This IS expected if you are initializing VideoLaVITLlamaForCausalLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing VideoLaVITLlamaForCausalLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


The Visual Vocab Size is 16384
The llama tokenizer vocab size is 32000
The maximal clip number is 16
Building Model Finsished


### Image Understanding

In [32]:
image_path = '/data/phd/wsun/datasets/test/01April_2010_Thursday_heute-6694_vertical.png'
prompt = """
Describe only the motion and gestures of the person in the image focus on hands and face.
"""

output = runner({"image": image_path, "text_input": prompt}, length_penalty=1, temperature=1.0, \
                use_nucleus_sampling=False, num_beams=1, truct_vqa=False, max_length=512,)[0]
print(output)


In the image, a person is standing in front of a striped background, with their hands positioned in front of their face. The person appears to be making a gesture, possibly covering their face or making a hand gesture. The striped background adds a visual element to the scene, creating a unique and interesting composition.


### Video Understanding

In [44]:
video_path = '/data/phd/wsun/datasets/test/train_video/15June_2010_Tuesday_tagesschau-86.mp4'
prompt = """
Describe the motion in this video.
Example:
Q: Describe the motion in this video.
A: Right hand: Raised to chest level, with fingers slightly bent. The palm
is facing the body.
Left hand: Remains still near the wairs, forming a fist.
Face: Concentrated expression with slightly furrowed brows and lips pressed together.

"""

output = runner({"video": video_path, "text_input": prompt}, length_penalty=1, \
        use_nucleus_sampling=False, num_beams=1, max_length=512, temperature=1.0)[0]
print(output)

The woman in the video is making a series of hand gestures, including raising her right hand to her chest and forming a fist with her left hand. Her face is focused and determined, and she appears to be speaking to the camera.


### Batch Video Understanding → JSON

遍历某一目录下所有 `.mp4`，逐个调用上面的 `runner`；结果写入一个 JSON 文件（列表），每项为：`id`（原视频文件名，不含路径）、`action_desc`（模型输出）。请先运行「加载模型」单元，再修改下一格中的 `VIDEO_DIR` / `OUTPUT_JSON` 后执行。


In [47]:
import json
import traceback
from pathlib import Path

# --------- 按需修改 ---------
VIDEO_DIR = Path("/data/phd/wsun/datasets/test/dev_video")  # 存放多个 mp4 的目录
OUTPUT_JSON = Path("/data/phd/wsun/datasets/test/Phoenix_lavit_v2ad_dev.json")
# id 使用「不含扩展名的文件名」；若需完整文件名（含 .mp4）可改为 id_key = "name"
id_key = "stem"  # "stem" | "name"

batch_prompt = """
Describe the motion in this video.
Example:
Q: Describe the motion in this video.
A: Right hand: Raised to chest level, with fingers slightly bent. The palm
is facing the body.
Left hand: Remains still near the wairs, forming a fist.
Face: Concentrated expression with slightly furrowed brows and lips pressed together.
"""
# ---------------------------

video_files = sorted(VIDEO_DIR.glob("*.mp4"))
if not video_files:
    raise FileNotFoundError(f"未在目录中找到 mp4: {VIDEO_DIR}")

results = []
for vp in video_files:
    vid = vp.name if id_key == "name" else vp.stem
    try:
        out = runner(
            {"video": str(vp), "text_input": batch_prompt},
            length_penalty=1,
            use_nucleus_sampling=False,
            num_beams=1,
            max_length=512,
            temperature=1.0,
        )[0]
        results.append({"id": vid, "action_desc": out})
        print(f"OK: {vid}")
    except Exception as e:
        err_txt = f"[ERROR] {type(e).__name__}: {e}"
        results.append({"id": vid, "action_desc": err_txt})
        print(f"FAIL: {vid}\n{traceback.format_exc()}")

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"已写入 {len(results)} 条 -> {OUTPUT_JSON}")


OK: 01April_2010_Thursday_heute-6697
OK: 01April_2010_Thursday_heute-6698
OK: 01December_2011_Thursday_heute-3058
OK: 01December_2011_Thursday_heute-3060
OK: 01December_2011_Thursday_heute-3061
OK: 01February_2011_Tuesday_heute-4637
OK: 01February_2011_Tuesday_tagesschau-4975
OK: 01February_2011_Tuesday_tagesschau-4976
OK: 01July_2009_Wednesday_tagesschau-4555
OK: 01July_2011_Friday_tagesschau-2763
OK: 01July_2011_Friday_tagesschau-2765
OK: 01July_2011_Friday_tagesschau-2766
OK: 01June_2010_Tuesday_tagesschau-5002
OK: 01March_2011_Tuesday_tagesschau-2209
OK: 01March_2011_Tuesday_tagesschau-2210
OK: 01March_2011_Tuesday_tagesschau-2211
OK: 01May_2010_Saturday_tagesschau-7197
OK: 01November_2010_Monday_tagesschau-140
OK: 01October_2009_Thursday_tagesschau-417
OK: 01October_2010_Friday_tagesschau-2691
OK: 02December_2010_Thursday_heute-8444
OK: 02December_2010_Thursday_heute-8445
OK: 02December_2010_Thursday_tagesschau-3636
OK: 02December_2010_Thursday_tagesschau-3637
OK: 02December_2011_